# Void Loss Analysis — Jupyter Lab

**Goal:** Recreate the Look Ahead interview project from scratch.

You will:
1. Load `Interview_Data_Worksheet.xlsx`
2. Understand its wide layout
3. Transform it into analysis-ready (long) format
4. Build summary tables
5. Export CSV files for Power BI

**Run cells top to bottom.** Change `DATA_DIR` in Step 1 if your folder path is different.

## Step 0 — Install packages (run once)

If you get `ModuleNotFoundError`, run this in a terminal from this folder:

```bash
cd "/Users/kwiffstaff/Downloads/Aks-CV/Look_ahead/Lookahead"
python3 -m venv .venv
source .venv/bin/activate
pip install pandas openpyxl jupyter
python -m ipykernel install --user --name=lookahead-venv
```

In Jupyter: **Kernel → Change Kernel → lookahead-venv** (or pick the venv Python).

In [ ]:
# Quick check that libraries are available
import pandas as pd
import numpy as np
from pathlib import Path

print(f"pandas {pd.__version__}")

## Step 1 — Set paths and load the raw Excel file

We read the sheet **without** treating row 0 as headers, because the real headers are on **row 2** (Excel row 3) and month dates are on **row 1**.

In [ ]:
# Folder containing Interview_Data_Worksheet.xlsx
DATA_DIR = Path("/Users/kwiffstaff/Downloads/Aks-CV/Look_ahead/Lookahead")

EXCEL_FILE = DATA_DIR / "Interview_Data_Worksheet.xlsx"

assert EXCEL_FILE.exists(), f"File not found: {EXCEL_FILE}"

# header=None keeps every row as data so we can inspect layout
raw = pd.read_excel(EXCEL_FILE, header=None)

print("Shape (rows, columns):", raw.shape)
raw.iloc[:5, :10]

## Step 2 — Understand the spreadsheet layout

| Row index | Excel row | Contents |
|-----------|-----------|----------|
| 0 | 1 | Empty / title area |
| 1 | 2 | **Month end dates** (repeated every 3 columns from col G) |
| 2 | 3 | **Column headers** (Building Ref, Building, … then Monthly Void Loss, Rent, %) |
| 3+ | 4+ | **One row per building** |

Columns **0–5** (A–F): building metadata  
Columns **6+**: repeating monthly blocks of **3 columns** each.

In [ ]:
# Row 2 = header labels for metadata + first month metrics
header_row = raw.iloc[2]
print("Metadata columns (0-5):")
for i in range(6):
    print(f"  Col {i}: {header_row.iloc[i]}")

# Row 1 = month dates — every 3rd column starting at 6
print("\nFirst 4 month blocks:")
for start_col in range(6, 6 + 4 * 3, 3):
    month_date = raw.iloc[1, start_col]
    metrics = raw.iloc[2, start_col : start_col + 3].tolist()
    print(f"  Col {start_col}: {month_date} -> {metrics}")

## Step 3 — Build a list of month column positions

We scan from column 6 in steps of 3 until we hit a blank date. Each block stores where to read void loss, rent, and %.

In [ ]:
months = []
col = 6

while col < raw.shape[1]:
    month_date = raw.iloc[1, col]
    if pd.isna(month_date):
        break
    months.append({
        "date": pd.Timestamp(month_date),
        "void_loss_col": col,
        "rent_col": col + 1,
        "pct_col": col + 2,
    })
    col += 3

print(f"Found {len(months)} months:")
for m in months:
    print(f"  {m['date'].strftime('%Y-%m')}")

## Step 4 — Unpivot: wide Excel → long dataframe

**Long format** = one row per **building × month**.  
This is the standard shape for Power BI and for `SUM(void) / SUM(rent)` KPIs.

In [ ]:
META_COLS = {
    0: "Building Ref",
    1: "Building",
    2: "Specialism",
    3: "Manager",
    4: "Local Authority",
    5: "Void Loss Target",
}

FIRST_DATA_ROW = 3  # row index where building data starts

records = []

for row_idx in range(FIRST_DATA_ROW, raw.shape[0]):
    building_ref = raw.iloc[row_idx, 0]
    if pd.isna(building_ref):
        continue  # skip blank rows

    meta = {name: raw.iloc[row_idx, col_idx] for col_idx, name in META_COLS.items()}
    meta["Void Loss Target"] = float(meta["Void Loss Target"])

    for m in months:
        void_loss = raw.iloc[row_idx, m["void_loss_col"]]
        rent = raw.iloc[row_idx, m["rent_col"]]
        pct = raw.iloc[row_idx, m["pct_col"]]

        records.append({
            **meta,
            "Month": m["date"],
            "Void Loss": float(void_loss) if pd.notna(void_loss) else 0.0,
            "Rent Charged": float(rent) if pd.notna(rent) else 0.0,
            "Void Loss Pct": float(pct) if pd.notna(pct) else np.nan,
        })

long_df = pd.DataFrame(records)

# MonthStart = first day of month (easier for charts than month-end dates)
long_df["MonthStart"] = long_df["Month"].dt.to_period("M").dt.to_timestamp()

print("Long table shape:", long_df.shape)
print("Buildings:", long_df["Building Ref"].nunique())
print("Months:", long_df["MonthStart"].nunique())
long_df.head()

## Step 5 — Check the KPI calculation (organisation YTD)

SMT KPI for the year to date:

$$\text{Void Loss \%} = \frac{\sum \text{Void Loss}}{\sum \text{Rent Charged}}$$

**Lower % is better.** Organisational target for 2025/26 = **6.5%**.

In [ ]:
ORG_TARGET = 0.065

ytd_void = long_df["Void Loss"].sum()
ytd_rent = long_df["Rent Charged"].sum()
ytd_pct = ytd_void / ytd_rent

print(f"YTD Void Loss £:  {ytd_void:,.2f}")
print(f"YTD Rent £:       {ytd_rent:,.2f}")
print(f"YTD Void Loss %:  {ytd_pct * 100:.2f}%")
print(f"Target:           {ORG_TARGET * 100:.1f}%")
print(f"Variance:         {(ytd_pct - ORG_TARGET) * 100:+.2f} pp")
print(f"Excess vs target: £{ytd_void - ORG_TARGET * ytd_rent:,.0f}")

## Step 6 — Organisation monthly summary

Group by month and sum £ across all buildings → in-month organisational performance.

In [ ]:
org_monthly = (
    long_df.groupby("MonthStart", as_index=False)
    .agg(Void_Loss=("Void Loss", "sum"), Rent=("Rent Charged", "sum"))
)
org_monthly["Pct"] = org_monthly["Void_Loss"] / org_monthly["Rent"]

org_monthly["Month"] = org_monthly["MonthStart"].dt.strftime("%b %Y")
org_monthly[["Month", "Void_Loss", "Rent", "Pct"]].assign(
    Pct=lambda d: (d["Pct"] * 100).round(2)
)

## Step 7 — Building YTD summary

For each building: sum void loss and rent over all months, then compute YTD % and whether they beat their **building target**.

In [ ]:
building_ytd = (
    long_df.groupby(
        ["Building Ref", "Building", "Specialism", "Manager", "Local Authority", "Void Loss Target"],
        as_index=False,
    )
    .agg(VL=("Void Loss", "sum"), Rent=("Rent Charged", "sum"))
)

building_ytd["YTD_Pct"] = building_ytd["VL"] / building_ytd["Rent"]
building_ytd["Beat_Target"] = building_ytd["YTD_Pct"] <= building_ytd["Void Loss Target"]

print(f"Buildings meeting target: {building_ytd['Beat_Target'].sum()} / {len(building_ytd)}")
print("\nWorst 5 buildings by YTD %:")
building_ytd.nlargest(5, "YTD_Pct")[["Building", "Specialism", "Void Loss Target", "YTD_Pct", "VL"]]

## Step 8 — Specialism YTD summary

In [ ]:
specialism_ytd = (
    long_df.groupby("Specialism", as_index=False)
    .agg(VL=("Void Loss", "sum"), Rent=("Rent Charged", "sum"))
)
specialism_ytd["YTD_Pct"] = specialism_ytd["VL"] / specialism_ytd["Rent"]
specialism_ytd["vs_target_pp"] = (specialism_ytd["YTD_Pct"] - ORG_TARGET) * 100

specialism_ytd.assign(
    YTD_Pct_pct=(specialism_ytd["YTD_Pct"] * 100).round(2),
    vs_target_pp=specialism_ytd["vs_target_pp"].round(2),
)[["Specialism", "VL", "Rent", "YTD_Pct_pct", "vs_target_pp"]]

## Step 9 — Export CSV files for Power BI

These four files match what we used in the Power BI build guide.

In [ ]:
OUTPUT_FILES = {
    "powerbi_void_loss_long.csv": long_df,
    "powerbi_org_monthly.csv": org_monthly,
    "powerbi_building_ytd.csv": building_ytd,
    "powerbi_specialism_ytd.csv": specialism_ytd,
}

for filename, df in OUTPUT_FILES.items():
    path = DATA_DIR / filename
    df.to_csv(path, index=False)
    print(f"Saved {path.name}  ({len(df)} rows)")

## Step 10 — Optional explorations (learning exercises)

Try these yourself by editing the cells below:
1. Plot monthly org % with a horizontal line at 6.5%
2. Bar chart of YTD % by Specialism
3. Filter `building_ytd` for one Manager and sort by void loss £

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(org_monthly["MonthStart"], org_monthly["Pct"] * 100, marker="o", label="Org void loss %")
ax.axhline(ORG_TARGET * 100, color="red", linestyle="--", label="Target 6.5%")
ax.set_ylabel("Void loss %")
ax.set_title("Organisation monthly void loss performance")
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## What you learned

| Concept | What you did |
|---------|----------------|
| **Wide vs long data** | Excel is wide (months as columns); analysis uses long (months as rows) |
| **Unpivot / melt** | Loop buildings × months to build `long_df` |
| **KPI aggregation** | Org YTD % = sum(void) / sum(rent), not average of monthly %s |
| **Groupby** | Monthly org, building YTD, specialism YTD |
| **Export** | CSV → Power BI for dashboards |

**Next steps:** Open `PowerBI_Build_Guide.md` and import the CSVs into Power BI Desktop.